# PhasePhyto Metrics Inspector

Loads saved JSON metrics for one run and compares PhasePhyto against the ViT baseline. Does not display images or reports.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    # Set this to a specific run folder name under drive_project_dir/runs.
    # If None, the latest run by folder name is used.
    "run_name": None,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + LOCATE RUN
# ============================================================

from pathlib import Path
import json

from google.colab import drive
from IPython.display import Image, Markdown, display

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path(CONFIG["drive_project_dir"])
RUNS_DIR = PROJECT_DIR / "runs"

if not RUNS_DIR.exists():
    raise RuntimeError(f"No runs directory found: {RUNS_DIR}")

if CONFIG["run_name"]:
    RUN_DIR = RUNS_DIR / CONFIG["run_name"]
else:
    candidates = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No run directories found under {RUNS_DIR}")
    RUN_DIR = candidates[-1]

MANIFEST_PATH = RUN_DIR / "run_manifest.json"
if not MANIFEST_PATH.exists():
    raise RuntimeError(f"No run_manifest.json found at {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text())
artifacts = manifest.get("artifacts", {})

print(f"Loaded run: {RUN_DIR}")
print(f"Manifest: {MANIFEST_PATH}")

try:
    import pandas as pd
except ImportError:
    pd = None


In [ ]:
# ============================================================
# LOAD METRICS
# ============================================================

results_path = Path(artifacts.get("results_json", RUN_DIR / "results" / "phasephyto_results.json"))
domain_path = Path(artifacts.get("domain_shift_json", RUN_DIR / "results" / "phasephyto_domain_shift.json"))

results = json.loads(results_path.read_text()) if results_path.exists() else {}
domain = json.loads(domain_path.read_text()) if domain_path.exists() else {}

print(f"Results JSON: {results_path} | exists={results_path.exists()}")
print(f"Domain JSON:  {domain_path} | exists={domain_path.exists()}")

summary = []
for model_name in ["baseline_vit", "phasephyto"]:
    if model_name in results:
        summary.append({"model": model_name, **results[model_name]})

if summary and pd is not None:
    display(pd.DataFrame(summary))
elif summary:
    print(json.dumps(summary, indent=2))
else:
    print("No model summary found in results JSON.")


In [ ]:
# ============================================================
# DERIVED COMPARISON
# ============================================================

if "baseline_vit" in results and "phasephyto" in results:
    bl = results["baseline_vit"]
    pp = results["phasephyto"]
    derived = {
        "target_f1_gain": pp["target_f1"] - bl["target_f1"],
        "target_acc_gain": pp["target_acc"] - bl["target_acc"],
        "phasephyto_gap_abs": abs(pp["target_acc"] - pp["source_acc"]),
        "baseline_gap_abs": abs(bl["target_acc"] - bl["source_acc"]),
    }
    print(json.dumps(derived, indent=2))
    display(Markdown(f"**PhasePhyto target-F1 gain over baseline:** `{derived['target_f1_gain']:+.4f}`"))
else:
    print("Need both baseline_vit and phasephyto results to compute derived comparison.")
